In [ ]:
import os.path
import datetime

# Google 라이브러리 import
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# AI 에이전트가 요청할 권한 범위 (읽기 전용)
# Gmail과 Calendar를 모두 테스트하기 위해 2개 포함
SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/calendar.readonly"
]

def main():
    creds = None
    
    # --- [인증 처리] ---
    # 'token.json' 파일은 사용자가 인증을 완료하면 자동으로 생성됩니다.
    # 이 파일은 'credentials.json'의 '열쇠'로 한 번 인증된 후의
    # '마스터 키' 역할을 하며, 다음 실행부터는 로그인을 생략하게 해줍니다.
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # 'token.json'이 없거나, 유효하지 않은 경우
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # [핵심] 'credentials.json'을 사용하여 사용자 인증 흐름을 시작합니다.
            flow = InstalledAppFlow.from_client_secrets_file(
                "../google_oauth_credentials.json", SCOPES
            )
            # 이 코드가 실행되면, 사용자 PC에서 웹 브라우저가 열립니다.
            creds = flow.run_local_server(port=0)
        
        # 다음 실행을 위해 'token.json' 파일로 인증 정보를 저장합니다.
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    print("--- 1. 인증 성공 ---")

    # --- [Gmail API 테스트] ---
    try:
        service_gmail = build("gmail", "v1", credentials=creds)
        
        # Gmail의 라벨(예: INBOX, SENT) 목록을 가져와 봅니다.
        results = service_gmail.users().labels().list(userId="me").execute()
        labels = results.get("labels", [])

        print("\n--- 2. Gmail API 테스트 (라벨 목록) ---")
        if not labels:
            print("Gmail에서 라벨을 찾을 수 없습니다.")
        else:
            print("성공! 처음 5개 라벨:")
            for label in labels[:5]:
                print(f"- {label['name']}")
    except HttpError as error:
        print(f"Gmail API 테스트 실패: {error}")

    # --- [Google Calendar API 테스트] ---
    try:
        service_calendar = build("calendar", "v3", credentials=creds)

        # 지금 시간부터 향후 5개의 이벤트를 가져와 봅니다.
        now = datetime.datetime.utcnow().isoformat() + "Z"  # 'Z'는 UTC 시간을 의미
        
        events_result = (
            service_calendar.events()
            .list(
                calendarId="primary", # 기본 캘린더
                timeMin=now,
                maxResults=5,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )
        events = events_result.get("items", [])

        print("\n--- 3. Google Calendar API 테스트 (향후 5개 일정) ---")
        if not events:
            print("향후 일정이 없습니다.")
        else:
            print("성공! 향후 5개 일정:")
            for event in events:
                start = event["start"].get("dateTime", event["start"].get("date"))
                print(f"- {start} | {event['summary']}")
    except HttpError as error:
        print(f"Calendar API 테스트 실패: {error}")


if __name__ == "__main__":
    main()